# 03 — Build Profiles and Execution Metrics

This notebook builds historical workload profiles and point-in-time execution features from the canonical Google subset execution dataset.

For each execution, historical features are calculated using only prior executions from the same `recurrence_key`. This prevents future information from leaking into pre-execution recommendations.

### Input
`data/processed/google_subset_v1/google_subset_executions_v1.parquet`

### Outputs
Historical workload profiles and execution-level metrics for downstream splitting, baseline generation, and recommendation evaluation.

## Imports and Paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# Find repository root
REPO_ROOT = Path.cwd()

while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

INPUT_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "google_subset_v1"
    / "google_subset_executions_v1.parquet"
)

OUTPUT_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "google_subset_v1"
)

print("Repository root:", REPO_ROOT)
print("Input:", INPUT_PATH)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Canonical execution dataset not found:\n{INPUT_PATH}\n"
        "Run 02_reconstructExecutions.ipynb first."
    )

executions = pd.read_parquet(INPUT_PATH)

print(f"Loaded {len(executions):,} executions")
print(f"Columns: {len(executions.columns)}")

executions.head()

Repository root: /Users/sofiasilva/Masters-project/masters-project
Input: /Users/sofiasilva/Masters-project/masters-project/data/processed/google_subset_v1/google_subset_executions_v1.parquet
Loaded 253,585 executions
Columns: 26


,execution_id,collection_id,instance_index,candidate_instance_key,episode_number,execution_start_time_us,execution_end_time_us,observation_count,machine_count,requested_cpu_ncu,...,runtime_sec,slack_cpu_ncu,slack_memory_normalized,user,collection_name,collection_logical_name,user_clean,logical_workload_name,recurrence_key,recurring_job_id
0,100286199||134||episode_1,100286199,134,100286199||134,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
1,100286199||250||episode_1,100286199,250,100286199||250,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
2,100286199||269||episode_1,100286199,269,100286199||269,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
3,100286199||321||episode_1,100286199,321,100286199||321,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
4,100286199||323||episode_1,100286199,323,100286199||323,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...


## Complete Schema

In [2]:
executions.info()
executions.columns.tolist()
executions[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
        "runtime_sec",
        "requested_cpu_ncu",
        "requested_memory_normalized",
        "peak_cpu_ncu",
        "peak_memory_normalized",
    ]
].head(10)

<class 'pandas.DataFrame'>
RangeIndex: 253585 entries, 0 to 253584
Data columns (total 26 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   execution_id                 253585 non-null  str    
 1   collection_id                253585 non-null  int64  
 2   instance_index               253585 non-null  int64  
 3   candidate_instance_key       253585 non-null  str    
 4   episode_number               253585 non-null  int64  
 5   execution_start_time_us      253585 non-null  Int64  
 6   execution_end_time_us        253585 non-null  Int64  
 7   observation_count            253585 non-null  int64  
 8   machine_count                253585 non-null  int64  
 9   requested_cpu_ncu            252811 non-null  float64
 10  requested_memory_normalized  252811 non-null  float64
 11  average_cpu_ncu              253585 non-null  float64
 12  average_memory_normalized    253585 non-null  float64
 13  peak_cpu_n

,execution_id,recurrence_key,execution_start_time_us,runtime_sec,requested_cpu_ncu,requested_memory_normalized,peak_cpu_ncu,peak_memory_normalized
0,100286199||134||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
1,100286199||250||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
2,100286199||269||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
3,100286199||321||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
4,100286199||323||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
5,100287787||122||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2248500000000,300.0,0.032410,0.007294,0.010040,0.003429
6,100288420||58||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,976200000000,300.0,0.045715,0.028870,0.013321,0.000992
7,10084165805||105||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,300.0,0.070435,0.041626,0.015488,0.005989
8,10084165805||112||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,300.0,0.070435,0.041626,0.015488,0.005989
9,10084165805||17||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,300.0,0.070435,0.041626,0.015488,0.005989


## Validate Canonical Schema

Verify that the execution dataset contains the fields required to construct historical workload profiles and point-in-time features.

In [3]:
required_columns = [
    "execution_id",
    "recurrence_key",
    "execution_start_time_us",
    "runtime_sec",
    "requested_cpu_ncu",
    "requested_memory_normalized",
    "peak_cpu_ncu",
    "peak_memory_normalized",
]

missing_columns = [
    col for col in required_columns
    if col not in executions.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required canonical columns: {missing_columns}"
    )

print("All required canonical columns are present.")

print("Execution-level integrity checks")
print("--------------------------------")

print(f"Rows:                    {len(executions):,}")
print(f"Unique execution IDs:    {executions['execution_id'].nunique():,}")
print(f"Unique recurrence keys:  {executions['recurrence_key'].nunique():,}")
print(f"Duplicate execution IDs: {executions['execution_id'].duplicated().sum():,}")
print(f"Missing recurrence keys: {executions['recurrence_key'].isna().sum():,}")
print(f"Invalid runtimes (<= 0): {(executions['runtime_sec'] <= 0).sum():,}")

All required canonical columns are present.
Execution-level integrity checks
--------------------------------
Rows:                    253,585
Unique execution IDs:    253,585
Unique recurrence keys:  2,280
Duplicate execution IDs: 0
Missing recurrence keys: 0
Invalid runtimes (<= 0): 0


## Order Executions Chronologically

Executions are grouped by `recurrence_key` and ordered by execution start time.

This ordering establishes the temporal sequence needed to construct historical features using only information available before each execution.

In [4]:
history = executions.sort_values(
    by=[
        "recurrence_key",
        "execution_start_time_us",
        "execution_id",
    ]
).reset_index(drop=True)

history[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
    ]
].head(20)

,execution_id,recurrence_key,execution_start_time_us
0,83212368472||1008||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
1,83212368472||1042||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
2,83212368472||1056||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
3,83212368472||1096||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
4,83212368472||1099||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
5,83212368472||1113||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
6,83212368472||1119||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
7,83212368472||1221||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
8,83212368472||1279||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
9,83212368472||1332||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000


## Build Point-in-Time History

Historical features must use only executions that started before the current execution.

Executions with the same `recurrence_key` and `execution_start_time_us` are treated as occurring at the same point in time. They cannot contribute historical information to one another.

In [5]:
# Count executions occurring at each workload/time point
time_groups = (
    history.groupby(
        ["recurrence_key", "execution_start_time_us"],
        sort=False
    )
    .size()
    .rename("executions_at_start_time")
    .reset_index()
)

# Number of executions strictly before each time point
time_groups["prior_execution_count"] = (
    time_groups.groupby("recurrence_key")["executions_at_start_time"]
    .cumsum()
    - time_groups["executions_at_start_time"]
)

history = history.merge(
    time_groups,
    on=["recurrence_key", "execution_start_time_us"],
    how="left",
    validate="many_to_one",
)

history[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
        "executions_at_start_time",
        "prior_execution_count",
    ]
].head(20)

,execution_id,recurrence_key,execution_start_time_us,executions_at_start_time,prior_execution_count
0,83212368472||1008||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
1,83212368472||1042||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
2,83212368472||1056||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
3,83212368472||1096||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
4,83212368472||1099||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
5,83212368472||1113||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
6,83212368472||1119||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
7,83212368472||1221||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
8,83212368472||1279||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0
9,83212368472||1332||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,34,0


In [6]:
print("Point-in-time history summary")
print("-----------------------------")

print(
    f"Executions with no prior history: "
    f"{(history['prior_execution_count'] == 0).sum():,}"
)

print(
    f"Executions with prior history:    "
    f"{(history['prior_execution_count'] > 0).sum():,}"
)

print(
    f"Maximum prior executions:         "
    f"{history['prior_execution_count'].max():,}"
)

print(
    f"Executions sharing a start time:  "
    f"{(history['executions_at_start_time'] > 1).sum():,}"
)

Point-in-time history summary
-----------------------------
Executions with no prior history: 80,801
Executions with prior history:    172,784
Maximum prior executions:         25,138
Executions sharing a start time:  252,013


## Build Historical Resource Features

Historical CPU, memory, and runtime statistics are calculated for each recurring workload.

For an execution starting at time `t`, the historical profile contains only executions from the same `recurrence_key` with start times strictly earlier than `t`.

The current execution and executions sharing the same start time are excluded from their own historical features.

In [9]:
time_point_summary = (
    history.groupby("recurrence_key")["execution_start_time_us"]
    .nunique()
)

print("Distinct start times per recurrence key")
print("---------------------------------------")
print(time_point_summary.describe())

print(
    "\nTotal recurrence/time groups:",
    history[
        ["recurrence_key", "execution_start_time_us"]
    ].drop_duplicates().shape[0]
)

Distinct start times per recurrence key
---------------------------------------
count    2280.000000
mean        2.139912
std         6.999918
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       274.000000
Name: execution_start_time_us, dtype: float64

Total recurrence/time groups: 4879


In [10]:
# Historical resource fields to profile
profile_fields = {
    "peak_cpu_ncu": "hist_peak_cpu",
    "peak_memory_normalized": "hist_peak_memory",
    "runtime_sec": "hist_runtime",
}

historical_profiles = []

for recurrence_key, group in history.groupby("recurrence_key", sort=False):
    group = group.sort_values("execution_start_time_us")

    # Process each distinct start time together so simultaneous
    # executions cannot contribute history to one another.
    prior_rows = []

    for start_time, current_rows in group.groupby(
        "execution_start_time_us",
        sort=True
    ):
        record = {
            "recurrence_key": recurrence_key,
            "execution_start_time_us": start_time,
            "prior_execution_count": len(prior_rows),
        }

        if prior_rows:
            prior_df = pd.concat(prior_rows, ignore_index=True)

            for field, prefix in profile_fields.items():
                values = prior_df[field].dropna()

                record[f"{prefix}_count"] = len(values)

                if len(values) > 0:
                    record[f"{prefix}_mean"] = values.mean()
                    record[f"{prefix}_std"] = values.std()
                    record[f"{prefix}_p50"] = values.quantile(0.50)
                    record[f"{prefix}_p95"] = values.quantile(0.95)
                else:
                    record[f"{prefix}_mean"] = np.nan
                    record[f"{prefix}_std"] = np.nan
                    record[f"{prefix}_p50"] = np.nan
                    record[f"{prefix}_p95"] = np.nan

        else:
            # Cold-start: no executions existed before this time point.
            for _, prefix in profile_fields.items():
                record[f"{prefix}_count"] = 0
                record[f"{prefix}_mean"] = np.nan
                record[f"{prefix}_std"] = np.nan
                record[f"{prefix}_p50"] = np.nan
                record[f"{prefix}_p95"] = np.nan

        historical_profiles.append(record)

        # Only after calculating the current profile do these executions
        # become available as history for later start times.
        prior_rows.append(
            current_rows[
                [
                    "peak_cpu_ncu",
                    "peak_memory_normalized",
                    "runtime_sec",
                ]
            ]
        )

historical_profiles = pd.DataFrame(historical_profiles)

print(
    f"Created {len(historical_profiles):,} "
    "point-in-time historical profiles."
)

historical_profiles.head()

Created 4,879 point-in-time historical profiles.


,recurrence_key,execution_start_time_us,prior_execution_count,hist_peak_cpu_count,hist_peak_cpu_mean,hist_peak_cpu_std,hist_peak_cpu_p50,hist_peak_cpu_p95,hist_peak_memory_count,hist_peak_memory_mean,hist_peak_memory_std,hist_peak_memory_p50,hist_peak_memory_p95,hist_runtime_count,hist_runtime_mean,hist_runtime_std,hist_runtime_p50,hist_runtime_p95
0,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,116400000000,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
2,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,359400000000,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
3,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,45600000000,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,1200900000000,1,1,0.062744,NaN,0.062744,0.062744,1,0.005478,NaN,0.005478,0.005478,1,300.0,<NA>,300.0,300.0
